# C6-pytorch — Practice p18 — Solution


Each row of the first weight matrix is the normal vector of one
required half-plane, with bias one for the centered square.  Four
hidden bits must all fire, so the output score is their sum minus four.
For the shifted rectangle, only the two $x_1$ offsets change.


In [ ]:
import numpy as np
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)
np.random.seed(20260804)

class DenseLayer(nn.Module):
    """Session 2's pinned dense layer: weight (out, in), bias (out,)."""

    def __init__(self, weight, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.as_tensor(weight), requires_grad=False)
        self.bias = nn.Parameter(torch.as_tensor(bias), requires_grad=False)

    def forward(self, x):
        return x @ self.weight.T + self.bias

class ThresholdGate(nn.Module):
    """Session 2's gate: 1 where x >= 0, else 0; owns no parameters."""

    def forward(self, x):
        return (x >= 0).to(x.dtype)


torch.manual_seed(20260804)
cloud = torch.rand(400, 2) * 4 - 2

W1 = torch.tensor([[-1.0, 0.0], [1.0, 0.0],
                   [0.0, -1.0], [0.0, 1.0]])
b1 = torch.tensor([1.0, 1.0, 1.0, 1.0])
W2 = torch.tensor([[1.0, 1.0, 1.0, 1.0]])
b2 = torch.tensor([-4.0])
square_net = nn.Sequential(DenseLayer(W1, b1), ThresholdGate(),
                           DenseLayer(W2, b2), ThresholdGate())
verdict = square_net(cloud).ravel()
square_direct = (cloud.abs().max(dim=1).values <= 1).to(cloud.dtype)
frac_agree_square = float((verdict == square_direct).to(cloud.dtype).mean())
n_scalars = 4 * 2 + 4 + 1 * 4 + 1

b1_rect = torch.tensor([2.0, 0.0, 1.0, 1.0])
rect_net = nn.Sequential(DenseLayer(W1, b1_rect), ThresholdGate(),
                         DenseLayer(W2, b2), ThresholdGate())
rect_verdict = rect_net(cloud).ravel()
rect_direct = ((cloud[:, 0] >= 0) & (cloud[:, 0] <= 2)
               & (cloud[:, 1].abs() <= 1)).to(cloud.dtype)
frac_agree_rect = float((rect_verdict == rect_direct).to(cloud.dtype).mean())

frac_agree_square, n_scalars, b1_rect, frac_agree_rect


One dense-plus-gate unit fires on a single half-plane.  A bounded
square is the intersection of four half-planes, not itself one
half-plane, so no single unit can recognize it.


### Answer check


In [ ]:
assert frac_agree_square == 1.0
assert n_scalars == 17
assert torch.equal(b1_rect, torch.tensor([2.0, 0.0, 1.0, 1.0]))
assert torch.equal(rect_net[0].weight, square_net[0].weight)
assert torch.equal(rect_net[2].weight, square_net[2].weight)
assert torch.equal(rect_net[2].bias, square_net[2].bias)
assert frac_agree_rect == 1.0
